<a href="https://colab.research.google.com/github/ALAMP198/TA2526/blob/main/Pr%C3%A1ctica2AntonioMP_de_es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En este caso, voy a generar un traductor alemán-español. Para ello, voy a usar el modelo propuesto en la primera parte de la práctica (facebook/mbart-large-50).

En primer lugar, cargo las librerías necesarias e inicio sesión en Hugging Face.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
!pip install transformers==4.51.2 datasets evaluate sacrebleu

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

En este caso, voy a utilizar el dataset ["Helsinki-NLP/opus_books", disponible en Hugging Face](https://huggingface.co/datasets/Helsinki-NLP/opus_books). Este dataset contiene textos alineados en multitud de idiomas.

En esta práctica, voy a generar un traductor alemán españok, así que selecciono el conjunto "de-es".

Este subconjunto contiene millones de cadenas, así que voy a limitar los datos de entrenamiento a 5000.

In [ ]:
dataset = load_dataset("Helsinki-NLP/opus_books", "de-es")
dataset["train"] = dataset["train"].select(range(5000))

El dataset solo contiene un split de entrenamiento y las cadenas en alemán y español aparecen dentro de un diccionario, por lo que es necesario dividir el split para crear uno de evaluación y procesar 'translation' para obtener las cadenas en ambos idiomas.

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 5000
    })
})

In [ ]:
def preprocess(examples):
    inputs = [ex["de"] for ex in examples["translation"]]
    targets = [ex["es"] for ex in examples["translation"]]
    return {"de": inputs, "es": targets}

dataset = dataset.map(preprocess, batched=True, remove_columns=["translation", "id"])

# Filtrar filas muy cortas (metadatos, títulos, etc.)
dataset = dataset.filter(lambda x: len(x["de"]) > 20 and len(x["es"]) > 20)

In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)


Vemos la apariencia del dataset procesado con Pandas.

In [ ]:
dataset['train'].to_pandas()



,de,es
0,"Ein- für allemal muß ich Ihnen nämlich sagen, ...","Conste, de una vez para siempre, que no quiero..."
1,"»Ihr Name ist Reed, Sir, Mrs. Reed.«",-Se llama Mrs. Reed y...
2,"Denn ich kannte Mr Rochester, bevor ich Madame...","Rochester antes que a Madame Frédéric, y me re..."
3,"»Stämmig – wirklich stämmig, Jane; groß, braun...","-Robusta, alta, morena, con un cabello como de..."
4,Außerdem war mir das Außergewöhnliche seines V...,"Por el contrario, semejante recepción me dejab..."
...,...,...
4052,Ich kann ihren Namen nicht so gut aussprechen ...,¡No puedo pronunciar su nombre!
4053,"»Das werde ich thun, Madame.","-De su parte, gracias..."
4054,Ich konnte den Sinn des Ganzen nicht verstehen...,Me parecía ver mis propios pensamientos en las...
4055,Ich erwartete beinahe eine Zurechtweisung für ...,Esperaba una contestación violenta a una maner...


A continuación, cargamos el modelo y tokenizamos. A diferencia del código de la primera parte de la práctica, hemos añadido los atributos tokenizer.src_lang y tokenizer.tgt_lang, y el parámetro text_target. De esta forma, el modelo sabe cuáles son los idiomas de trabajo y cuál es la traducción esperada.

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
max_length = 128

tokenizer.src_lang = "de_DE"
tokenizer.tgt_lang = "es_XX"

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["de"],
        text_target=examples["es"],
        max_length=max_length,
        truncation=True,
    )
    return model_inputs

In [ ]:
tokenized_datasets = dataset.map(preprocess_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['de', 'es', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4057
    })
    test: Dataset({
        features: ['de', 'es', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 451
    })
})

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer)

Mantengo los mismos argumentos de la primera parte de la práctica, solo modifico el directorio de salida

In [ ]:
from transformers import Seq2SeqTrainingArguments


args = Seq2SeqTrainingArguments(
    # Carpeta donde se guardará el modelo
    output_dir="mbart-de-es",
    # Evaluar el modelo cada época
    eval_strategy="epoch",
    # Valores para modificar los pesos del modelo
    learning_rate=5.6e-5,
    weight_decay=0.01,
    # Número de ejemplos que se muestran cada vez al modelo
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    # Número de veces que que se muestran todos los datos al modelo
    num_train_epochs=2,
    # Realizar predicciones al validar
    predict_with_generate=True,
    push_to_hub=True,
    report_to="none"
)

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

In [ ]:
import numpy as np

import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,3.057400,2.456557,7.592700,27.421300
2,1.814700,2.420823,8.168500,28.266100


TrainOutput(global_step=1016, training_loss=2.426389844398799, metrics={'train_runtime': 2187.385, 'train_samples_per_second': 3.709, 'train_steps_per_second': 0.464, 'total_flos': 1558985325821952.0, 'train_loss': 2.426389844398799, 'epoch': 2.0})

Como podemos comprobar, los valores de Bleu son muy bajos, lo que indica que el modelo no es preciso. Esto se debe a que se ha utilizado un dataset breve con una capacidad de cómputo baja.

Subo el modelo a Hugging Face.

In [ ]:
%cd mbart-de-es
trainer.push_to_hub(commit_message="Training complete", tags="simplification")

/content/mbart-neutralization


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-de-es/training_args.bin: 100%|##########| 5.84kB / 5.84kB            

  ...s/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...t-de-es/model.safetensors:   0%|          | 7.98MB / 2.44GB            

  ...bart-de-es/tokenizer.json:  47%|####6     | 7.97MB / 17.1MB            

CommitInfo(commit_url='https://huggingface.co/ntr2026/mbart-de-es/commit/e03cd76c8a0643a64b64006de5de7042da51e829', commit_message='Training complete', commit_description='', oid='e03cd76c8a0643a64b64006de5de7042da51e829', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ntr2026/mbart-de-es', endpoint='https://huggingface.co', repo_type='model', repo_id='ntr2026/mbart-de-es'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch

model = MBartForConditionalGeneration.from_pretrained('ntr2026/mbart-de-es')
tokenizer = MBart50TokenizerFast.from_pretrained('ntr2026/mbart-de-es')

def traducir(texto):
    tokenizer.src_lang = "de_DE"
    inputs = tokenizer(texto, return_tensors="pt", padding=True, truncation=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["es_XX"],
            max_length=128
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(traducir("Guten Morgen, wie geht es dir?"))

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

-Buenos días, ¿cómo te encuentras?


In [ ]:
print(neutralizar("Was denkst du darüber?"))

¿Qué piensas tú de ella?
